# Sub-seasonal Weather Forecast Pipeline

GraphCast / FourCastNet / MetNet-3 によるアンサンブル予測 + メタ学習統合

## 使い方
1. Google Driveに `weather-forecast/` フォルダごとアップロード
2. ランタイム → ランタイムのタイプを変更 → GPU (A100推奨)
3. 「すべてのセルを実行」または各セルを順次実行

## Cell 1: 環境セットアップ

In [ ]:
# Google Driveマウント
from google.colab import drive
drive.mount('/content/drive')

# パス設定
import sys
PROJECT_DIR = '/content/drive/MyDrive/weather-forecast/colab'
sys.path.insert(0, PROJECT_DIR)

print(f'Project directory: {PROJECT_DIR}')

In [ ]:
# 依存ライブラリのインストール
!pip install -q \
  cdsapi \
  ecmwf-opendata \
  xarray \
  cfgrib \
  netcdf4 \
  dm-haiku \
  jax \
  jaxlib \
  lightgbm \
  optuna \
  torch \
  supabase \
  numpy \
  scipy

# GraphCast (DeepMind)
!pip install -q git+https://github.com/google-deepmind/graphcast.git

# FourCastNet (NVIDIA earth2mip)
!pip install -q earth2mip

print('All dependencies installed.')

## Cell 2: 環境変数の設定

以下のAPIキーを設定してください。

In [ ]:
import os
from google.colab import userdata

# Colab Secrets から取得（推奨）
# 左サイドバー 🔑 → 各キーを登録してください
try:
    os.environ['CDS_API_KEY'] = userdata.get('CDS_API_KEY')
    os.environ['SUPABASE_URL'] = userdata.get('SUPABASE_URL')
    os.environ['SUPABASE_KEY'] = userdata.get('SUPABASE_KEY')
    print('Secrets loaded from Colab Secrets.')
except Exception:
    print('Could not load from Colab Secrets. Set environment variables manually below:')
    # 手動設定 (Secrets が使えない場合)
    # os.environ['CDS_API_KEY'] = 'your-cds-api-key'
    # os.environ['SUPABASE_URL'] = 'https://xxxxx.supabase.co'
    # os.environ['SUPABASE_KEY'] = 'your-supabase-anon-key'

## Cell 3: 予測パラメータの設定

In [ ]:
# === 予測パラメータ ===
TARGET_DATE = '2026-04-15'  # 予測対象日
REGION = 'japan'            # 対象領域
N_MEMBERS = 50              # アンサンブルメンバー数

print(f'Target date: {TARGET_DATE}')
print(f'Region: {REGION}')
print(f'Ensemble members: {N_MEMBERS}')

## Step 1: データ取得

In [ ]:
from src.data_fetcher import fetch_all_data

print('Fetching meteorological data...')
raw_data = fetch_all_data(target_date=TARGET_DATE, region=REGION)

# 取得結果の確認
for source, ds in raw_data.items():
    if ds is not None:
        print(f'  {source}: {list(ds.data_vars)} ({ds.dims})')
    else:
        print(f'  {source}: not available')

## Step 2: アンサンブル推論

In [ ]:
from src.ensemble_inference import run_ensemble

print(f'Running ensemble inference ({N_MEMBERS} members x 3 models)...')
ensemble_results = run_ensemble(raw_data, n_members=N_MEMBERS)

# 各モデルのアンサンブル統計を確認
for model_name, stats in ensemble_results.items():
    print(f'\n  {model_name}:')
    for stat_name, values in stats.items():
        if hasattr(values, 'shape'):
            print(f'    {stat_name}: shape={values.shape}')
        else:
            print(f'    {stat_name}: {type(values)}')

## Step 3: メタ学習統合 + 天気概況変換

In [ ]:
from src.meta_learner import run_meta_learning

print('Running meta-learning integration...')
forecasts = run_meta_learning(ensemble_results)

# 結果のプレビュー
import pandas as pd
df = pd.DataFrame(forecasts)
print(f'\nGenerated {len(forecasts)} forecast(s):')
display(df[['date', 'location', 'weather', 'temp_max', 'temp_min', 'precipitation_prob', 'confidence']])

## Step 4: Supabaseアップロード

In [ ]:
from src.upload import upload_to_supabase

print('Uploading forecasts to Supabase...')
result = upload_to_supabase(forecasts)

print(f'Upload complete.')
print(f'  Run ID: {result["run_id"]}')
print(f'  Forecasts uploaded: {result["count"]}')

---
## Done!

予測結果がSupabaseにアップロードされました。
Next.jsアプリ (`/forecast`) から予測結果を確認し、Googleカレンダーに同期できます。